In [ ]:
# ==============================================================================
# 📊 [AlexNet 논문 Section 3.1 및 Figure 1 재현]
# 활성화 함수 비교 실험: ReLU vs Tanh vs Sigmoid 수렴 속도 및 기울기 소실 분석
# Paper Quote: "Deep convolutional neural networks with ReLUs train several times faster than their equivalents with tanh units."
# ==============================================================================

import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 1. 구글 드라이브 마운트 및 데이터셋 로드
try:
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_dir = '/content/drive/MyDrive/cifar10_data'
    print(f"📌 구글 드라이브 데이터 경로: {dataset_dir}")
except ImportError:
    dataset_dir = './data'
    print("📌 로컬 환경 실행: ./data 디렉토리를 사용합니다.")

os.makedirs(dataset_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 현재 디바이스: {device}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

trainset = torchvision.datasets.CIFAR10(root=dataset_dir, train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root=dataset_dir, train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)
print("✅ 데이터셋 준비 완료!")


# 2. 활성화 함수 변경 가능 AlexNet 모델 클래스
class AlexNetActivationExp(nn.Module):
    def __init__(self, activation='relu', num_classes=10):
        super(AlexNetActivationExp, self).__init__()
        self.activation_type = activation
        self.lrn = nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0)
        
        self.conv1 = nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2)
        
        self.conv2 = nn.Conv2d(96, 256, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2)
        
        self.conv3 = nn.Conv2d(256, 384, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(384, 384, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(384, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=3, stride=2)
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.Linear(4096, num_classes)
        )

    def _act(self, x):
        if self.activation_type == 'relu':
            return F.relu(x)
        elif self.activation_type == 'tanh':
            return torch.tanh(x)
        elif self.activation_type == 'sigmoid':
            return torch.sigmoid(x)
        return x

    def forward(self, x):
        x = self.pool1(self.lrn(self._act(self.conv1(x))))
        x = self.pool2(self.lrn(self._act(self.conv2(x))))
        
        x = self._act(self.conv3(x))
        x = self._act(self.conv4(x))
        x = self.pool3(self._act(self.conv5(x)))
        
        x = torch.flatten(x, 1)
        feats = self.classifier[:2](x)
        feats = self._act(feats)
        feats = self.classifier[2:4](feats)
        feats = self._act(feats)
        return self.classifier[4](feats)


# 3. 활성화 함수별 학습 및 소요 시간 측정을 위한 헬퍼 함수
def train_and_measure(act_name, epochs=2):
    print(f"\n🏋️ [{act_name.upper()}] 기반 AlexNet 학습 시작...")
    model = AlexNetActivationExp(activation=act_name).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0005)
    
    losses = []
    start_time = time.time()
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            if i % 15 == 14:
                avg_loss = running_loss / 15
                losses.append(avg_loss)
                print(f"[{act_name.upper()}] Epoch [{epoch+1}/{epochs}], Batch [{i+1}/{len(trainloader)}], Loss: {avg_loss:.4f}")
                running_loss = 0.0
                
    elapsed_time = time.time() - start_time
    print(f"✅ [{act_name.upper()}] 학습 소요 시간: {elapsed_time:.2f}초")
    return losses, elapsed_time


# 4. 3가지 활성화 함수 (ReLU, Tanh, Sigmoid) 실험 실행
losses_relu, time_relu = train_and_measure('relu', epochs=2)
losses_tanh, time_tanh = train_and_measure('tanh', epochs=2)
losses_sig, time_sig = train_and_measure('sigmoid', epochs=2)


# 5. [시각화 1] 논문 Figure 1 스타일 손실 수렴 비교 그래프 (Loss Convergence)
plt.figure(figsize=(10, 5))
plt.plot(losses_relu, label='AlexNet with ReLU (Paper Baseline: Fast Convergence)', color='crimson', linewidth=2.8)
plt.plot(losses_tanh, label='AlexNet with Tanh (Saturating Nonlinearity)', color='royalblue', linewidth=2.5, linestyle='--')
plt.plot(losses_sig, label='AlexNet with Sigmoid (Severe Vanishing Gradient)', color='orange', linewidth=2.5, linestyle=':')

plt.title('AlexNet Paper Section 3.1 & Figure 1: ReLU vs Tanh vs Sigmoid Convergence', fontsize=13, fontweight='bold')
plt.xlabel('Training Iterations (x15 batches)', fontsize=11)
plt.ylabel('Training Loss (Cross Entropy)', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# 6. [시각화 2] 총 소요 시간 비교 바 차트 (Training Time Comparison)
plt.figure(figsize=(8, 4.5))
act_labels = ['ReLU', 'Tanh', 'Sigmoid']
times = [time_relu, time_tanh, time_sig]
colors = ['crimson', 'royalblue', 'orange']

bars = plt.bar(act_labels, times, color=colors, width=0.5, alpha=0.85)
plt.title('Total Training Time per Activation Function (2 Epochs)', fontsize=13, fontweight='bold')
plt.ylabel('Elapsed Time (Seconds)', fontsize=11)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f"{yval:.1f}s", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("🎉 ReLU vs Tanh vs Sigmoid 비교 실험 완료!")

